# Simple Text Classifier Project

In this hands-on project, you'll build a simple text classifier using Python and scikit-learn. We'll use a dataset of SMS messages labeled as spam or ham (not spam).

**Steps:**
1. Load and explore the data
2. Preprocess the text
3. Extract features (TF-IDF)
4. Train a classifier (Naive Bayes)
5. Evaluate the model
6. Draw conclusions


## 1. Load and Explore the Data
We'll use the SMS Spam Collection dataset from UCI, which is available via openml in scikit-learn.

In [ ]:
import pandas as pd
import os
import requests
import zipfile
import tempfile

from sklearn.datasets import fetch_openml

## Problem with sklearn.datasets fetch_openml? No problem:
Depends on your fetch_openml's version, but this method couldn't works, therefore I created a direct function able to download the spam dataset from UCI website. Check here below:

In [8]:
def load_sms_spam_dataset_from_uci():
    print("Trying UCI direct download...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp_file:
        tmp_file.write(response.content)
        tmp_file_path = tmp_file.name

    with zipfile.ZipFile(tmp_file_path, 'r') as zip_ref:
        zip_ref.extractall(tempfile.gettempdir())

    # Read the SMS data
    sms_file = os.path.join(tempfile.gettempdir(), 'SMSSpamCollection')
    df = pd.read_csv(sms_file, sep='\t', header=None, names=['class', 'text'])

    # Clean up
    os.unlink(tmp_file_path)
    os.unlink(sms_file)

    print("✅ Successfully loaded from UCI!")

    return df

In [12]:

try:
# Load the SMS Spam Collection dataset
    print("Trying fetch_openml...")
    data = fetch_openml('sms_spam', version=1, as_frame=True)
    df = data.frame
except:
    try:
        df = load_sms_spam_dataset_from_uci()
    except Exception as e:
        print(f"Error loading dataset: {e}")

df.head()

Trying fetch_openml...
Trying UCI direct download...
✅ Successfully loaded from UCI!


,class,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 2. Preprocess the Text
Let's clean the text data. We'll lowercase, remove punctuation, and strip whitespace.

In [13]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = text.strip()
    return text

df['text_clean'] = df['text'].apply(clean_text)
df[['text', 'text_clean']].head()

,text,text_clean
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


## 3. Feature Extraction (TF-IDF)
Convert the cleaned text into numerical features using TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['text_clean'])
y = df['class']
X.shape

## 4. Train/Test Split
Split the data into training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## 5. Train a Naive Bayes Classifier
We'll use Multinomial Naive Bayes, which is well-suited for text data.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

## 6. Evaluate the Model
Let's check the accuracy and see a confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))

## 7. Conclusion
You have built a simple text classifier using Python and scikit-learn!

- You learned how to preprocess text, extract features, train a model, and evaluate it.
- Try experimenting with different models or preprocessing steps to improve performance.